# 特許調査パイプライン（ステップ1・2）
1. J-PlatPat の検索結果一覧（コピーしたテキスト）から文献番号表を作る
2. 文献番号から Google Patents（日本語ページ）の HTML を取得し、名称・要約・請求項・明細書を Excel に出力する

**使い方**: 上から順に実行。まず設定セルの値を変更し、手順2は `LIMIT = 3` などで少数件を試してから全件にしてください。

注意: Google Patents の自動取得は利用規約上の制限を受け得ます。`DELAY` を空けて、必要な件数だけ実行してください。

In [ ]:
%pip install -q pandas openpyxl requests beautifulsoup4 lxml

## 設定

In [ ]:
from pathlib import Path

INPUT_TXT = "patent_data1.txt"          # J-PlatPat 結果一覧のコピーテキスト
NUMBERS_XLSX = "results/patent_numbers.xlsx"   # 手順1の出力
RESULT_XLSX = "results/patents_text.xlsx"      # 手順2の出力
HTML_DIR = "results/html"               # 取得した生 HTML の保存先（再実行時は取得済みを飛ばす）
LIMIT = 3                               # 手順2で処理する件数。None なら全件
DELAY = 3.0                             # 取得間隔(秒)

Path("results").mkdir(exist_ok=True)

## 手順1: 文献番号の抽出（コード）

In [ ]:
"""J-PlatPat「検索結果一覧(国内文献)」のコピーテキストを表に変換する。

J-PlatPat の画面は JavaScript で描画されるため、保存した HTML には表が入らない。
結果一覧をコピーして貼り付けたテキストを読み、文献番号などを取り出す。
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field, asdict
from pathlib import Path

FI_RE = re.compile(r"^[A-HY]\d{2}[A-Z]\d+")
DATE_RE = re.compile(r"^\d{4}/\d{2}/\d{2}$")
DOC_RE = re.compile(
    r"^(?P<kind>特開|特表|再表|特許|実登|実公|実開|公開実用|登録実用|特公)"
    r"(?P<era>昭|平)?(?P<body>[\d\-/]+)$"
)
# 文献種別 -> Google Patents の種別コード（推定。B1/B2 や Y1/Y2 は判別できない）
KIND_CODE = {"特開": "A", "特表": "A", "再表": "A1", "特許": "B2",
             "実登": "U", "実開": "U", "公開実用": "U", "登録実用": "U",
             "実公": "Y2", "特公": "B2"}
STATUS_STARTS = ("審査", "特許", "（出願の）", "-", "登録", "権利", "実用", "拒絶")
HEADER_END = "FI"


@dataclass
class PatentRow:
    no: int
    doc_no: str = ""
    app_no: str = ""
    filing_date: str = ""
    publication_date: str = ""
    title: str = ""
    applicant: str = ""
    applicant_has_more: bool = False
    status: list[str] = field(default_factory=list)
    fi: list[str] = field(default_factory=list)
    fi_has_more: bool = False
    warnings: list[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        d = asdict(self)
        d["status"] = " / ".join(self.status)
        d["fi"] = " ; ".join(self.fi)
        d["warnings"] = " ; ".join(self.warnings)
        d["google_patent_id"] = google_patent_id(self.doc_no)
        d["google_patent_url"] = (
            f"https://patents.google.com/patent/{d['google_patent_id']}/ja"
            if d["google_patent_id"] else ""
        )
        return d


def google_patent_id(doc_no: str) -> str:
    """文献番号から Google Patents の ID 候補を作る（種別コードは推定）。

    例: 特開2026-139649 -> JP2026139649A / 特開昭50-051518 -> JPS50051518A
    """
    m = DOC_RE.match(doc_no)
    if not m:
        return ""
    if m["kind"] == "再表":  # 例: 再表2015/072306 -> JPWO2015072306A1, 再表92/019759 -> JPWO1992019759A1
        y, n = m["body"].split("/")
        y = y if len(y) == 4 else ("19" if int(y) >= 50 else "20") + y
        return f"JPWO{y}{n}A1"
    era = {"昭": "S", "平": "H", None: ""}[m["era"]]
    digits = m["body"].replace("-", "")
    return f"JP{era}{digits}{KIND_CODE[m['kind']]}"


def _is_status_line(line: str) -> bool:
    return line.startswith(STATUS_STARTS)


def parse_text(text: str) -> list[PatentRow]:
    lines = [l.strip() for l in text.splitlines()]
    lines = [l for l in lines if l]
    # ヘッダ（"FI" まで）を飛ばす
    try:
        start = lines.index(HEADER_END) + 1
    except ValueError:
        start = 0
    body = lines[start:]

    # レコード境界: 連番 1,2,3... と一致する数字だけの行
    # （文献番号は数字だけにならないため、連番との一致で十分に区別できる）
    starts, expect = [], 1
    for i, l in enumerate(body):
        if l.isdigit() and int(l) == expect:
            starts.append(i)
            expect += 1
    rows = []
    for k, s in enumerate(starts):
        e = starts[k + 1] if k + 1 < len(starts) else len(body)
        chunk = body[s + 1:e]
        if k + 1 == len(starts):  # 末尾のフッタを除去
            chunk = [l for l in chunk if not l.startswith(("Copyright", "(P"))]
        rows.append(_parse_record(k + 1, chunk))
    return rows


def _parse_record(no: int, chunk: list[str]) -> PatentRow:
    r = PatentRow(no=no)
    if len(chunk) < 6:
        r.warnings.append("行数不足")
        return r
    r.doc_no, r.app_no, r.filing_date, r.publication_date, r.title, r.applicant = chunk[:6]
    rest = chunk[6:]
    if not DOC_RE.match(r.doc_no):
        r.warnings.append("文献番号の書式が未知")
    for label, v in (("出願日", r.filing_date), ("公知日", r.publication_date)):
        if not DATE_RE.match(v):
            r.warnings.append(f"{label}の書式が未知")
    # 出願人欄の「他」（FI の「他」と区別するため、FI より前にあるものだけ）
    i = 0
    if rest and rest[0] == "他":
        r.applicant_has_more = True
        i = 1
    for l in rest[i:]:
        if l == "他":
            r.fi_has_more = True
        elif FI_RE.match(l) or re.match(r"^[A-HY]\d{2}[A-Z]\d+", l):
            r.fi.append(l)
        elif _is_status_line(l) or not r.fi:
            r.status.append(l)
        else:
            r.warnings.append(f"分類不能な行: {l}")
    return r


def load_file(path: str | Path) -> list[PatentRow]:
    return parse_text(Path(path).read_text(encoding="utf-8"))


## 手順1: 実行

In [ ]:
import pandas as pd

rows = load_file(INPUT_TXT)
COLUMNS = {
    "no": "No.", "doc_no": "文献番号", "app_no": "出願番号", "filing_date": "出願日",
    "publication_date": "公知日", "title": "発明の名称", "applicant": "出願人/権利者",
    "applicant_has_more": "出願人(他あり)", "status": "ステータス", "fi": "FI",
    "fi_has_more": "FI(他あり)", "google_patent_id": "Google Patents ID(推定)",
    "google_patent_url": "Google Patents URL(推定)", "warnings": "警告",
}
numbers = pd.DataFrame([r.to_dict() for r in rows])[list(COLUMNS)].rename(columns=COLUMNS)
numbers.to_excel(NUMBERS_XLSX, index=False)
print(f"{len(numbers)} 件 -> {NUMBERS_XLSX} / 警告あり {int((numbers['警告'] != '').sum())} 件")
numbers.head(10)

In [ ]:
numbers[numbers['警告'] != ''][['No.', '文献番号', '警告']]  # 読み取れなかった行（空なら問題なし）

## 手順2: Google Patents から取得・抽出（コード）

In [ ]:
"""文献番号の表(Excel)から Google Patents(日本語ページ)のHTMLを取得し、本文を抽出して Excel に出力する。

python -m patent_search.fetch_google IN.xlsx OUT.xlsx [--html-dir results/html] [--limit N] [--delay 3]

- 取得した生の HTML は --html-dir に保存する（再実行時は取得済みをスキップ）。
- 本文は <section itemprop=...> / <div class="claim"> 等の構造から抽出する。
  Google 側の画面構造は予告なく変わるため、抽出に失敗した行は「抽出状況」に記録する。
- 注意: Google Patents の自動取得は利用規約上の制限を受け得る。件数を絞り、間隔を空けること。
"""

from __future__ import annotations

import argparse
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = "https://patents.google.com/patent/{id}/ja"
UA = "Mozilla/5.0 (patent_search research; contact: owner of this repository)"
EXCEL_CELL_MAX = 32000


def _text(node) -> str:
    """ブロック内のインライン要素(<u>, <sub> 等)は改行せず連結し、<br> だけを改行にする。"""
    if node is None:
        return ""
    node = BeautifulSoup(str(node), "lxml")
    for br in node.find_all("br"):
        br.replace_with("\n")
    text = node.get_text("")
    lines = (re.sub(r"[ \t\u3000]+", " ", l).strip() for l in text.splitlines())
    return "\n".join(l for l in lines if l)


def _content(section):
    """見出し(<h2>)を除いた本体。"""
    return section.find(attrs={"itemprop": "content"}) or section if section else None


def extract(html: str) -> dict:
    s = BeautifulSoup(html, "lxml")
    title = s.find(attrs={"itemprop": "title"}) or s.find("h1")
    abstract = s.find("section", attrs={"itemprop": "abstract"})
    desc = s.find("section", attrs={"itemprop": "description"})
    claims_sec = s.find("section", attrs={"itemprop": "claims"})
    # 請求項は <li class="claim"> の中の <div class="claim" num="N"> が実体（li と div の二重取りを避ける）
    claims = []
    if claims_sec:
        for c in claims_sec.find_all("div", class_="claim"):
            claims.append(f"【請求項{c.get('num', len(claims) + 1)}】" + _text(c))
    paras = []
    if desc:
        for p in desc.find_all(class_="description-paragraph"):
            paras.append(f"[{p.get('num', '')}] " + _text(p) if p.get("num") else _text(p))
    out = {
        "title": re.sub(r"\s+", " ", _text(title)).replace(" - Google Patents", ""),
        "abstract": _text(_content(abstract)),
        "claims": "\n".join(claims),
        "description": "\n".join(paras) or _text(_content(desc)),
        "claim_count": len(claims),
    }
    missing = [k for k in ("title", "abstract", "claims", "description") if not out[k]]
    out["extract_status"] = "OK" if not missing else "欠落: " + ",".join(missing)
    return out


def fetch(session: requests.Session, pid: str, html_dir: Path, delay: float):
    """(html, 使った ID, HTTP 状態) を返す。ID は 末尾の種別コードを外した候補も試す。"""
    for cand in dict.fromkeys([pid, re.sub(r"[A-Z]\d?$", "", pid)]):
        path = html_dir / f"{cand}.html"
        if path.exists():
            return path.read_text(encoding="utf-8"), cand, "cache"
        time.sleep(delay)
        r = session.get(URL.format(id=cand), timeout=60, headers={"Accept-Language": "ja"})
        if r.status_code == 200:
            r.encoding = "utf-8"
            path.write_text(r.text, encoding="utf-8")
            return r.text, cand, "200"
        status = str(r.status_code)
    return "", pid, status


def run(df: pd.DataFrame, html_dir: Path, delay: float = 3.0) -> pd.DataFrame:
    """文献番号の表(Google Patents ID(推定) 列を持つ)から取得・抽出し、結果の表を返す。"""
    html_dir = Path(html_dir)
    html_dir.mkdir(parents=True, exist_ok=True)
    sess = requests.Session()
    sess.headers["User-Agent"] = UA
    rows = []
    for _, r in df.iterrows():
        pid = r.get("Google Patents ID(推定)")
        rec = {"文献番号": r["文献番号"], "ID": pid}
        if not isinstance(pid, str) or not pid:
            rec.update({"取得状況": "IDなし"})
        else:
            try:
                html, used, status = fetch(sess, pid, html_dir, delay)
            except requests.RequestException as e:
                html, used, status = "", pid, f"error: {type(e).__name__}"
            rec.update({"使用ID": used, "取得状況": status})
            if html:
                x = extract(html)
                rec.update({"名称(Google)": x["title"], "要約": x["abstract"],
                            "請求項数": x["claim_count"], "請求項": x["claims"],
                            "明細書": x["description"], "抽出状況": x["extract_status"]})
                if len(x["description"]) > EXCEL_CELL_MAX:  # Excel のセル上限対策。全文は HTML 側にある
                    rec["明細書"] = x["description"][:EXCEL_CELL_MAX]
                    rec["抽出状況"] += " / 明細書を切り詰め"
                if len(x["claims"]) > EXCEL_CELL_MAX:
                    rec["請求項"] = x["claims"][:EXCEL_CELL_MAX]
                    rec["抽出状況"] += " / 請求項を切り詰め"
        rows.append(rec)
        print(rec["文献番号"], rec.get("取得状況"), rec.get("抽出状況", ""), flush=True)
    return pd.DataFrame(rows)


## 手順2: 実行
まず `LIMIT = 3` 程度で、取得状況・抽出状況を確認してください。

In [ ]:
targets = numbers if LIMIT is None else numbers.head(LIMIT)
result = run(targets, Path(HTML_DIR), DELAY)
result.to_excel(RESULT_XLSX, index=False)
print('->', RESULT_XLSX)
result[[c for c in ['文献番号', 'ID', '使用ID', '取得状況', '抽出状況', '請求項数'] if c in result]]

取得状況が `200`/`cache` 以外、または抽出状況が `OK` 以外の行を確認します。

In [ ]:
result[(result['取得状況'].isin(['200', 'cache']) == False) | (result.get('抽出状況', 'OK') != 'OK')]